In [194]:
# EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import chi2, chi2_contingency

# ML
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

### Carga de Dados

In [58]:
df_leads = pd.read_csv('./datasets/leads.csv')

In [97]:
df_leads.info()

<class 'pandas.DataFrame'>
Index: 9074 entries, 0 to 9239
Data columns (total 17 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Lead Origin                             9074 non-null   str    
 1   Lead Source                             9074 non-null   str    
 2   Do Not Email                            9074 non-null   int64  
 3   Do Not Call                             9074 non-null   int64  
 4   Converted                               9074 non-null   int64  
 5   TotalVisits                             9074 non-null   float64
 6   Total Time Spent on Website             9074 non-null   int64  
 7   Page Views Per Visit                    9074 non-null   float64
 8   Last Activity                           9074 non-null   str    
 9   Search                                  9074 non-null   int64  
 10  Newspaper Article                       9074 non-null   int64  
 11  X Educa

In [96]:
df_leads.head(10)

,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Last Activity,Search,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,A free copy of Mastering The Interview,Last Notable Activity
0,API,Olark Chat,0,0,0,0.0,0,0.0,Page Visited on Website,0,0,0,0,0,0,0,Modified
1,API,Organic Search,0,0,0,5.0,674,2.5,Email Opened,0,0,0,0,0,0,0,Email Opened
2,Landing Page Submission,Direct Traffic,0,0,1,2.0,1532,2.0,Email Opened,0,0,0,0,0,0,1,Email Opened
3,Landing Page Submission,Direct Traffic,0,0,0,1.0,305,1.0,Unreachable,0,0,0,0,0,0,0,Modified
4,Landing Page Submission,Google,0,0,1,2.0,1428,1.0,Converted to Lead,0,0,0,0,0,0,0,Modified
5,API,Olark Chat,0,0,0,0.0,0,0.0,Olark Chat Conversation,0,0,0,0,0,0,0,Modified
6,Landing Page Submission,Google,0,0,1,2.0,1640,2.0,Email Opened,0,0,0,0,0,0,0,Modified
7,API,Olark Chat,0,0,0,0.0,0,0.0,Olark Chat Conversation,0,0,0,0,0,0,0,Modified
8,Landing Page Submission,Direct Traffic,0,0,0,2.0,71,2.0,Email Opened,0,0,0,0,0,0,1,Email Opened
9,API,Google,0,0,0,4.0,58,4.0,Email Opened,0,0,0,0,0,0,0,Email Opened


In [95]:
df_leads.tail(10)

,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Last Activity,Search,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,A free copy of Mastering The Interview,Last Notable Activity
9230,Landing Page Submission,Google,0,0,0,2.0,870,2.00,Email Opened,0,0,0,0,0,0,0,Email Opened
9231,Landing Page Submission,Google,0,0,1,8.0,1016,4.00,Email Opened,0,0,0,0,0,0,0,Email Opened
9232,Landing Page Submission,Direct Traffic,0,0,0,2.0,1770,2.00,SMS Sent,0,0,0,0,0,0,1,SMS Sent
9233,API,Direct Traffic,0,0,1,13.0,1409,2.60,SMS Sent,0,0,0,0,0,0,0,SMS Sent
9234,Landing Page Submission,Direct Traffic,0,0,1,5.0,210,2.50,SMS Sent,0,0,0,0,0,0,0,Modified
9235,Landing Page Submission,Direct Traffic,1,0,1,8.0,1845,2.67,Email Marked Spam,0,0,0,0,0,0,0,Email Marked Spam
9236,Landing Page Submission,Direct Traffic,0,0,0,2.0,238,2.00,SMS Sent,0,0,0,0,0,0,1,SMS Sent
9237,Landing Page Submission,Direct Traffic,1,0,0,2.0,199,2.00,SMS Sent,0,0,0,0,0,0,1,SMS Sent
9238,Landing Page Submission,Google,0,0,1,3.0,499,3.00,SMS Sent,0,0,0,0,0,0,0,SMS Sent
9239,Landing Page Submission,Direct Traffic,0,0,1,6.0,1279,3.00,SMS Sent,0,0,0,0,0,0,1,Modified


### Feature Engineering e Data Cleaning

In [62]:
df_leads.drop(['Prospect ID', 'Lead Number'], axis=1, inplace=True)

In [63]:
# Remove colunas com apenas um valor único

for column in df_leads.select_dtypes('object'):
    if df_leads[column].nunique() == 1:
        print(f"{column}: {df_leads[column].unique}")

        df_leads.drop(column, axis=1, inplace=True)

Magazine: <bound method Series.unique of 0       No
1       No
2       No
3       No
4       No
        ..
9235    No
9236    No
9237    No
9238    No
9239    No
Name: Magazine, Length: 9240, dtype: str>
Receive More Updates About Our Courses: <bound method Series.unique of 0       No
1       No
2       No
3       No
4       No
        ..
9235    No
9236    No
9237    No
9238    No
9239    No
Name: Receive More Updates About Our Courses, Length: 9240, dtype: str>
Update me on Supply Chain Content: <bound method Series.unique of 0       No
1       No
2       No
3       No
4       No
        ..
9235    No
9236    No
9237    No
9238    No
9239    No
Name: Update me on Supply Chain Content, Length: 9240, dtype: str>
Get updates on DM Content: <bound method Series.unique of 0       No
1       No
2       No
3       No
4       No
        ..
9235    No
9236    No
9237    No
9238    No
9239    No
Name: Get updates on DM Content, Length: 9240, dtype: str>
I agree to pay the amount through cheque

/tmp/ipykernel_15765/3997046926.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_leads.select_dtypes('object'):


In [ ]:
# Mostando e removendo colunas categóricas com percentual de valores ausentes acima de 25%

for column in df_leads.select_dtypes(include='object'):
    contagem_nulas = (df_leads[column] == 'Select').sum() + df_leads[column].isnull().sum()

    if contagem_nulas / len(df_leads) * 100 > 25:
        print(f'{column}: {contagem_nulas / len(df_leads) * 100:.2f}%')

        df_leads.drop(column, axis=1, inplace=True)

Country: 26.63%
Specialization: 36.58%
How did you hear about X Education: 78.46%
What is your current occupation: 29.11%
What matters most to you in choosing a course: 29.32%
Tags: 36.29%
Lead Quality: 51.59%
Lead Profile: 74.19%
City: 39.71%
Asymmetrique Activity Index: 45.65%
Asymmetrique Profile Index: 45.65%


/tmp/ipykernel_15765/4101281906.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_leads.select_dtypes(include='object'):


In [79]:
df_leads['Lead Source'] = df_leads['Lead Source'].apply(lambda x: 'Google' if x == 'google' else x)

In [ ]:
# Transformando colunas com valores Booleanos categóricos para Binário

for column in df_leads.select_dtypes(include='object'):
    valores_unicos = df_leads[column].unique()

    if set(valores_unicos).issubset(set(['Yes', 'No'])):
        print(f'{column}')

        df_leads[column] = df_leads[column].apply((lambda x: 1 if x == 'Yes' else 0))

Do Not Email
Do Not Call
Search
Newspaper Article
X Education Forums
Newspaper
Digital Advertisement
Through Recommendations
A free copy of Mastering The Interview


/tmp/ipykernel_15765/420006417.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_leads.select_dtypes(include='object'):


In [ ]:
# Removendo colunas categóricas com valores ausentes

colunas_categoricas = df_leads.select_dtypes(include='object').columns
df_leads.dropna(subset=colunas_categoricas, inplace=True)

/tmp/ipykernel_15765/2161987220.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_categoricas = df_leads.select_dtypes(include='object').columns


In [ ]:
# Mostando e removendo colunas numéricas com percentual de valores ausentes acima de 25%

for column in df_leads.select_dtypes(include='number'):
    contagem_nulas = df_leads[column].isnull().sum()

    if contagem_nulas / len(df_leads) * 100 > 25:
        print(f'{column}: {contagem_nulas / len(df_leads) * 100:.2f}%')

        df_leads.drop(column, axis=1, inplace=True)

Asymmetrique Activity Score: 45.69%
Asymmetrique Profile Score: 45.69%


In [ ]:
# Removendo colunas numéricas com valores ausentes

colunas_numericas = df_leads.select_dtypes(include='number').columns
df_leads.dropna(subset=colunas_numericas, inplace=True)

### EDA

In [98]:
df_leads.describe()

,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Search,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,A free copy of Mastering The Interview
count,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000
mean,0.078907,0.000220,0.378554,3.456028,482.887481,2.370151,0.001543,0.000220,0.000110,0.000110,0.000441,0.000771,0.318272
std,0.269608,0.014845,0.485053,4.858802,545.256560,2.160871,0.039251,0.014845,0.010498,0.010498,0.020992,0.027766,0.465831
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,1.000000,11.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,3.000000,246.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,1.000000,5.000000,922.750000,3.200000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,251.000000,2272.000000,55.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [103]:
df_leads['Converted'].value_counts(normalize=True)

Converted
0    0.621446
1    0.378554
Name: proportion, dtype: float64

In [ ]:
# Plotar a distribuição dos valores da feature target

fig = px.bar(df_leads['Converted'].value_counts(normalize=True) * 100, 
            title='Hit Ratio - Fator de Conversão',
            labels={'index': 'Converted', 'value': 'Percentual'},
            opacity=0.8)

fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Matriz de correlação das variáveis numéricas
corr_matrix = df_leads.select_dtypes(include='number').corr()

In [120]:
# Plot de correlação

fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        x = corr_matrix.columns,
        y = corr_matrix.index,
        z = np.array(corr_matrix),
        text = corr_matrix.values,
        texttemplate = '%{text:.2f}',
        colorscale = px.colors.diverging.RdBu,
        zmin = -1,
        zmax = 1
    )
)

fig.show()

In [ ]:
# BoxPlot - Converted x TotalVisits
fig = px.box(df_leads, x='Converted',  y='TotalVisits', color='Converted')
fig.show()

In [123]:
# BoxPlot - Converted x Total Time Spent on Website
fig = px.box(df_leads, x='Converted',  y='Total Time Spent on Website', color='Converted')
fig.show()

In [124]:
# BoxPlot - Converted x Page Views Per Visit
fig = px.box(df_leads, x='Converted',  y='Page Views Per Visit', color='Converted')
fig.show()

In [125]:
# Tabela de Contigẽncia de Converted x Lead Source
contigency_table_lead_source = pd.crosstab(df_leads['Converted'], df_leads['Lead Source'])

In [126]:
contigency_table_lead_source

Lead Source,Click2call,Direct Traffic,Facebook,Google,Live Chat,NC_EDM,Olark Chat,Organic Search,Pay per Click Ads,Press_Release,Reference,Referral Sites,Social Media,WeLearn,Welingak Website,bing,blog,testone,welearnblog_Home,youtubechannel
Converted,,,,,,,,,,,,,,,,,,,,
0,1,1725,22,1726,0,0,1305,718,1,2,33,94,1,0,2,5,1,1,1,1
1,3,818,9,1147,2,1,448,436,0,0,410,31,1,1,127,1,0,0,0,0


In [ ]:
# Teste de Independência de qui-quadrado

chi2, p, dof, expected = chi2_contingency(contigency_table_lead_source)

print(f'Estatística qui-quadrado: {chi2}')
print(f'Valor p: {p}')
print(f'Graus de Liberdade: {dof}')
print(f'Existe uma relação significativa entre Converted e Lead Source? {p < 0.05}')

Estatística qui-quadrado: 942.1372507753774
Valor p: 1.1748671316223743e-187
Graus de Liberdade: 19
Existe uma relação significativa entre Converted e Lead Source? True


In [160]:
# Tabela de Contigẽncia de Converted x Lead Origin
contigency_table_lead_origin = pd.crosstab(df_leads['Converted'], df_leads['Lead Origin'])

In [161]:
contigency_table_lead_origin

Lead Origin,API,Landing Page Submission,Lead Add Form,Lead Import
Converted,,,,
0,2463,3118,37,21
1,1115,1767,544,9


In [172]:
# Teste de Independência de qui-quadrado

chi2, p, dof, expected = chi2_contingency(contigency_table_lead_origin)

print(f'Estatística qui-quadrado: {chi2}')
print(f'Valor p: {p}')
print(f'Graus de Liberdade: {dof}')
print(f'Existe uma relação significativa entre Converted e Lead Origin? {p < 0.05}')

Estatística qui-quadrado: 1424.6171966295433
Valor p: 8.365508263958168e-295
Graus de Liberdade: 15
Existe uma relação significativa entre Converted e Lead Origin? True


In [168]:
# Tabela de Contigẽncia de Converted x Last Notable Activity
contigency_table_last_notable_activity = pd.crosstab(df_leads['Converted'], df_leads['Last Notable Activity'])

In [169]:
contigency_table_last_notable_activity

Last Notable Activity,Approached upfront,Email Bounced,Email Link Clicked,Email Marked Spam,Email Opened,Email Received,Form Submitted on Website,Had a Phone Conversation,Modified,Olark Chat Conversation,Page Visited on Website,Resubscribed to emails,SMS Sent,Unreachable,Unsubscribed,View in browser link Clicked
Converted,,,,,,,,,,,,,,,,
0,0,51,128,0,1781,0,1,1,2587,158,225,0,663,10,33,1
1,1,9,45,2,1042,1,0,13,680,25,93,1,1489,22,12,0


In [171]:
# Teste de Independência de qui-quadrado

chi2, p, dof, expected = chi2_contingency(contigency_table_last_notable_activity)

print(f'Estatística qui-quadrado: {chi2}')
print(f'Valor p: {p}')
print(f'Graus de Liberdade: {dof}')
print(f'Existe uma relação significativa entre Converted e Last Notable Activity? {p < 0.05}')

Estatística qui-quadrado: 1424.6171966295433
Valor p: 8.365508263958168e-295
Graus de Liberdade: 15
Existe uma relação significativa entre Converted e Last Notable Activity? True


### Preparação dos Dados

In [ ]:
X = df_leads.drop('Converted', axis=1)
y = df_leads['Converted']

In [176]:
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

/tmp/ipykernel_15765/1130913818.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include='object').columns


In [183]:
# Divisão de treino e teste

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=51)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [184]:
print(X_train.shape)
print(X_test.shape)

(7259, 68)
(1815, 68)


### Treinamento do Modelo

In [187]:
bagging_model = BaggingClassifier(
    estimator=LogisticRegression(),
    n_estimators=10,
    random_state=51
)

In [188]:
bagging_model.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",LogisticRegression()
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",51
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",10
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary <warm_start>`... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


### Avaliação do Modelo

In [ ]:
# Predição do conjunto de testes
y_pred = bagging_model.predict(X_test)

In [ ]:
# Avaliar modelo
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

In [193]:
print(f'Acurácia: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1-Score: {f1}')

Acurácia: 0.7972451790633609
Precision: 0.7467320261437909
Recall: 0.682089552238806
F1-Score: 0.7129485179407177


In [195]:
# Matriz de confusão
conf_matrix = confusion_matrix(y_test, y_pred)

fig = px.imshow(conf_matrix,
                labels=dict(x='Predição', y='Real', color='Contagem'),
                x=['Not Converted', 'Converted'],
                y=['Not Converted', 'Converted'],
                color_continuous_scale='Viridis')

fig.update_traces(text=conf_matrix, texttemplate='%{z}')
fig.update_layout(coloraxis_showscale=False)

fig.show()